# Python 클래스 파일별 2 · 2.ipynb self · 인스턴스 변수 · 클래스 변수

**한 줄 요약** · `self`가 메서드를 부른 객체라는 것과, 객체마다 따로인 인스턴스 변수 · 모두 공유하는 클래스 변수를 확인하고 `nn.Module`의 `self.층`으로 연결

- **원본** · `2.ipynb/01_강의노트/2.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/02_Python클래스` 전체 정리 2장 · 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `car1.show_info()`가 `Car.show_info(car1)`로 바뀌어 실행됨을 설명한다
2. 메서드 정의에서 `self`를 빠뜨리면 생기는 에러를 말한다
3. 인스턴스 변수(객체마다)와 클래스 변수(공유)를 구분하고, 공유 값은 `클래스.변수`로 바꾼다
4. `nn.Module`의 `self.fc`가 모델 객체마다 따로인 인스턴스 변수임을 확인한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">모델 안에 층을 저장할 때</td>
<td style="text-align: left;"><code>self.fc = nn.Linear(...)</code> 인스턴스 변수 (3장)</td>
</tr>
<tr>
<td style="text-align: left;">데이터셋마다 다른 목록을 가질 때</td>
<td style="text-align: left;"><code>self.items = items</code> (2-1)</td>
</tr>
<tr>
<td style="text-align: left;">모든 객체가 같이 쓰는 상수 · 개수</td>
<td style="text-align: left;">클래스 변수 <code>Car.total_created</code> (2-2)</td>
</tr>
<tr>
<td style="text-align: left;"><code>model(x)</code>가 어떤 모델의 층을 쓰는지 이해할 때</td>
<td style="text-align: left;"><code>self</code> = 부른 객체 (1-1)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfc2_map.png" alt="2.ipynb 학습 지도" width="763">

**예상 학습 시간** · 개념 50분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 구성</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">self의 의미</td>
<td style="text-align: left;">메서드 안의 <code>self</code>는 누구이며, 빠뜨리면 어떻게 되는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">인스턴스 변수와 클래스 변수</td>
<td style="text-align: left;">객체마다 따로인 값과 모두 공유하는 값은 어떻게 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;"><code>nn.Module</code> 모델의 <code>self.fc</code>도 객체마다 따로인 인스턴스 변수인가?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

1 · 2장은 기본 문법만, 3장은 PyTorch를 씀. **셀은 위에서부터 차례로 실행**(1-1의 `Car` · `car1`을 2장 끝까지 씀).

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">PyTorch (3장만)</td>
</tr>
<tr>
<td style="text-align: left;">seed · 장치</td>
<td style="text-align: left;">3장 <code>torch.manual_seed(0)</code> · CPU</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">원본의 자동차 객체 · 3장은 <code>torch.ones</code> 입력</td>
</tr>
</tbody>
</table>

In [1]:
# 실행 환경 확인: 1 · 2장은 기본 문법, 3장만 PyTorch 사용
import sys  # 파이썬 버전 확인용

import torch  # 3장 nn.Module 실습

print("Python:", sys.version.split()[0])            # 3.10 이상이면 됨
print("PyTorch:", torch.__version__.split("+")[0])  # 설치 확인

Python: 3.14.6
PyTorch: 2.11.0


### 원본 구성

- 원본 학습 목표 3개
  - `self`가 현재 객체를 뜻한다는 것을 설명
  - 인스턴스 변수와 클래스 변수의 차이 설명
  - 여러 객체가 독립된 상태와 공유 상태를 갖는 모습 확인

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본 절</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1 ~ 3</td>
<td style="text-align: left;">객체 생성 · self의 의미 · 객체가 서로 다른지</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">4 ~ 5</td>
<td style="text-align: left;">인스턴스 변수의 독립성 · <code>__dict__</code></td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">6 ~ 7</td>
<td style="text-align: left;">클래스 변수 · 객체 추가 생성</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">8 ~ 10</td>
<td style="text-align: left;">같은 이름을 객체에 저장 · 삭제 · <code>isinstance</code></td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">보충</td>
<td style="text-align: left;">self를 빠뜨리면 · <code>nn.Module</code>에서 확인</td>
<td style="text-align: left;">1-2 · 3</td>
</tr>
</tbody>
</table>

- 원본의 객체 번호(`id`) 출력은 실행마다 달라 `is` 비교로 바꿈

**원본 힌트**

- 원본은 같은 `Car` 객체 4 ~ 5개로 self · 인스턴스 변수 · 클래스 변수를 차례로 확인
- 원본 출력의 생성 수 `127` · `193`은 셀을 반복 실행해 누적된 값 → 이 노트북은 처음부터 한 번씩 실행한 값
- `nn.Module`을 이해하려면 이 파일과 파일별 3(8.ipynb)이 특히 중요

---
## 1. self의 의미

> **이 장의 흐름**
>
> - 원본 1 ~ 3절: 객체를 만들고 두 가지 방식으로 메서드 호출
> - 보충: 메서드 정의에서 `self`를 빠뜨린 경우

<img src="images/cls2_self.png" alt="car1.show_info()는 Car.show_info(car1)로 실행" width="763">

**파이프라인 위치** · ④ 모델 — `model(x)`의 self

### 1-1. `A` 객체 생성과 self · `self` · `클래스.메서드(객체)` <sub>원본 1 ~ 3절</sub>

> 🎯 **`car1.show_info()`를 부르면 `self`에는 무엇이 들어가는가?**

💡 **쉽게 말하면**

- `self`는 **'이 메서드를 부른 바로 그 객체'**.
- `car1.show_info()`는 파이썬이 `Car.show_info(car1)`로 바꿔 실행.

**실행 흐름**

1. Car 정의 (클래스 변수 `total_created`)
2. 객체 4개 생성
3. 두 가지 호출 방식
4. is 로 서로 다른지

**먼저 예측** · `car1.show_info()`와 `Car.show_info(car1)`의 결과는 같을까? — 답은 코드 실행 결과 아래에

In [2]:
class Car:                                       # 생성 수를 세는 자동차 설계도
    """자동차 한 대의 정보를 표현하는 클래스"""  # 클래스 설명 (docstring)

    total_created = 0  # 클래스 변수: 모든 Car 객체가 함께 쓰는 값

    def __init__(self, company, color, price):  # 객체를 만들 때 자동 실행
        self.company = company                  # 인스턴스 변수: 객체마다 다른 값
        self.color = color                      # 색
        self.price = price                      # 가격
        Car.total_created += 1                  # 공유 값은 클래스 이름으로 변경

    def show_info(self):                                            # self = 이 메서드를 부른 객체
        return f"{self.company} / {self.color} / {self.price:,}원"  # 그 객체의 값으로 문장


car1 = Car("Ferrari", "White", 8000)  # 1번째 생성
car2 = Car("BMW", "Black", 5000)      # 2번째
car3 = Car("Audi", "Silver", 6000)    # 3번째
car4 = Car("Audi", "Silver", 18)      # 원본 1절의 4번째
print(car1.show_info())               # 파이썬이 car1을 self에 넣음
print(Car.show_info(car1))            # 직접 car1을 self 자리에
print("car1 is car2:", car1 is car2)  # 같은 클래스여도 다른 객체
# __str__ 없으면 주소 (실행마다 다름)
print("print(car1) →", str(car1).split(" at ")[0], "at 0x...>")

Ferrari / White / 8,000원
Ferrari / White / 8,000원
car1 is car2: False
print(car1) → <__main__.Car object at 0x...>


**결과** · 예측 정답은 같음 — `car1.show_info()`는 `Car.show_info(car1)`로 바뀌어 실행

- `Ferrari / White / 8,000원` — 두 호출 모두 같은 줄 (2번 출력)
- `car1 is car2: False` — 같은 설계도로 만든 다른 객체
- `print(car1) → <__main__.Car object at 0x...>` — `__str__`이 없으면 클래스 이름 + 주소

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>car1.show_info()</code></td>
<td style="text-align: left;">객체가 <strong>자동으로</strong> self에 들어감</td>
<td style="text-align: left;"><strong>기본 선택</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>Car.show_info(car1)</code></td>
<td style="text-align: left;">self 자리에 객체를 <strong>직접</strong> 넣음</td>
<td style="text-align: left;">동작 원리 확인용</td>
</tr>
<tr>
<td style="text-align: left;"><code>car1 is car2</code></td>
<td style="text-align: left;">같은 객체인지</td>
<td style="text-align: left;">같은 설계도라도 False</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `self`는 예약어가 아니라 관례 이름 → 다른 이름도 되지만 항상 `self`로 씀

> ✅ **한 줄 정리** · `self`는 메서드를 부른 객체이며, `객체.메서드()`는 `클래스.메서드(객체)`로 실행된다.

**딥러닝 코드에서는**

- `model(x)` → `forward(self, x)`의 self = 그 model 객체

### 1-2. `B` self를 빠뜨리면 · `def f():` · `TypeError` <sub>보충</sub>

> 🎯 **메서드를 `def hello():`처럼 `self` 없이 만들면 어떻게 되는가?**

💡 **쉽게 말하면**

- `객체.메서드()`는 **객체를 첫 인자로 몰래 넘김**.
- 받을 칸(self)이 없으면 인자가 하나 남아 TypeError.

**실행 흐름**

1. self 없는 메서드 정의
2. 객체로 부르기
3. 클래스로 부르기

**먼저 예측** · `NoSelf().hello()`는 인자를 몇 개 넘긴 것으로 보일까? — 답은 코드 실행 결과 아래에

In [3]:
class NoSelf:        # self를 빠뜨린 설계도
    def hello():     # 첫 인자 self 없음
        return "hi"  # 인사


try:
    NoSelf().hello()                       # 객체가 자동으로 첫 인자로 들어감
except TypeError as e:                     # 받을 칸이 없음
    print("TypeError:", e)                 # 에러 문장
print("클래스로 부르면:", NoSelf.hello())  # 객체가 안 넘어가 동작

TypeError: NoSelf.hello() takes 0 positional arguments but 1 was given
클래스로 부르면: hi


**결과** · 예측 정답은 1개 — 객체 자신

- `TypeError: NoSelf.hello() takes 0 positional arguments but 1 was given` — 객체가 첫 인자로 넘어갔는데 받을 칸이 없음
- `클래스로 부르면: hi` — 클래스로 부르면 객체가 안 넘어가 동작 (하지만 객체 값을 못 씀)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>def f(self):</code></td>
<td style="text-align: left;">객체를 self로 받음</td>
<td style="text-align: left;"><strong>기본 선택</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>def f():</code></td>
<td style="text-align: left;">객체로 부르면 TypeError</td>
<td style="text-align: left;">쓰면 안 됨</td>
</tr>
<tr>
<td style="text-align: left;"><code>@staticmethod</code></td>
<td style="text-align: left;">self 없이도 객체로 부를 수 있게 표시</td>
<td style="text-align: left;">객체 값이 필요 없는 도우미 함수</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 에러 문장의 `takes 0 ... but 1 was given`은 대부분 self 누락 신호

> ✅ **한 줄 정리** · 메서드는 첫 인자로 `self`를 받아야 하며, 빠뜨리면 객체로 부를 때 TypeError가 난다.

**딥러닝 코드에서는**

- `def forward(x):`로 쓰면 `model(x)`에서 같은 TypeError

---
## 2. 인스턴스 변수와 클래스 변수

> **이 장의 흐름**
>
> - 원본 4 ~ 7절: 인스턴스 변수의 독립성 · 클래스 변수 공유
> - 원본 8 ~ 10절: 같은 이름을 객체에 저장하면 · 삭제 · `isinstance`

<img src="images/pf_cls_vars.png" alt="car1 · car2의 인스턴스 변수와 공유 클래스 변수" width="763">

**파이프라인 위치** · ④ 모델 — 층(객체마다) · 상수(공유)

### 2-1. `A` 인스턴스 변수의 독립성 · `self.price` · `__dict__` <sub>원본 4 ~ 5절</sub>

> 🎯 **`car1.price`를 바꾸면 `car2.price`도 바뀌는가?**

💡 **쉽게 말하면**

- 인스턴스 변수는 **각자의 사물함**.
- 내 사물함을 바꿔도 친구 사물함은 그대로.

**실행 흐름**

1. 가격 변경 전
2. car1.price = 7500
3. 가격 변경 후
4. `__dict__` 확인

**먼저 예측** · `car1.price = 7500` 뒤 `car2.price`는? — 답은 코드 실행 결과 아래에

In [4]:
print("변경 전:", car1.price, car2.price)  # 8000 5000
car1.price = 7500                          # car1 객체의 값만 바꿈
print("변경 후:", car1.price, car2.price)  # car2는 그대로
print("car1:", car1.__dict__)              # car1에 직접 저장된 값
print("car2:", car2.__dict__)              # car2에 직접 저장된 값

변경 전: 8000 5000
변경 후: 7500 5000
car1: {'company': 'Ferrari', 'color': 'White', 'price': 7500}
car2: {'company': 'BMW', 'color': 'Black', 'price': 5000}


**결과** · 예측 정답은 `5000` 그대로

- `변경 전: 8000 5000` · `변경 후: 7500 5000` — car1만 바뀜
- `car1: {'company': 'Ferrari', 'color': 'White', 'price': 7500}` · `car2: {'company': 'BMW', 'color': 'Black', 'price': 5000}` — `__dict__` = 객체마다 따로 저장된 인스턴스 변수 (total_created는 없음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">인스턴스 변수 <code>self.price</code></td>
<td style="text-align: left;"><strong>객체마다</strong> 따로 (<code>__dict__</code>)</td>
<td style="text-align: left;">이름 · 가격 · 층 · 데이터 목록</td>
</tr>
<tr>
<td style="text-align: left;"><code>객체.__dict__</code></td>
<td style="text-align: left;">그 객체에 직접 저장된 값</td>
<td style="text-align: left;">무엇이 저장됐는지 확인</td>
</tr>
<tr>
<td style="text-align: left;"><code>car1.price = 값</code></td>
<td style="text-align: left;">car1의 값만 변경</td>
<td style="text-align: left;">객체 하나만 바꿀 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `__dict__`에 클래스 변수는 안 보임 → 클래스 변수는 `Car.__dict__` 쪽에 있음

> ✅ **한 줄 정리** · 인스턴스 변수는 객체마다 따로 저장되므로 한 객체를 바꿔도 다른 객체는 그대로다.

**딥러닝 코드에서는**

- `self.items = items` — 데이터셋 객체마다 다른 목록

### 2-2. `A` 클래스 변수 · `Car.total_created` · `공유` <sub>원본 6 ~ 7절</sub>

> 🎯 **클래스 변수는 어디에 있고, 누가 읽을 수 있는가?**

💡 **쉽게 말하면**

- 클래스 변수는 **반 전체 게시판**.
- 누가 보든 같은 값, 고칠 때는 반 이름(`Car.`)으로.

**실행 흐름**

1. 생성 수 확인
2. 객체로도 읽기
3. car4 다시 생성
4. 공유 값 확인

**먼저 예측** · 지금까지 4대를 만들었으면 `Car.total_created`는? 하나 더 만들면? — 답은 코드 실행 결과 아래에

In [5]:
print("생성한 자동차 수:", Car.total_created)  # car1 ~ car4
print("car1에서 접근:", car1.total_created)    # 객체로도 읽을 수 있음
car4 = Car("Volvo", "Blue", 4500)              # 7절: 하나 더 생성
print("car4 생성 후:", Car.total_created, "/ car2에서:", car2.total_created)  # 모두 같은 값을 봄

생성한 자동차 수: 4
car1에서 접근: 4
car4 생성 후: 5 / car2에서: 5


**결과** · 예측 정답은 `4` → `5`

- `생성한 자동차 수: 4` · `car1에서 접근: 4` — 클래스에 하나 → 어디서 읽어도 같은 값
- `car4 생성 후: 5 / car2에서: 5` — 새 객체가 늘린 값을 모두가 봄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">클래스 변수 <code>Car.total_created</code></td>
<td style="text-align: left;">클래스에 <strong>1개</strong>, 모두 공유</td>
<td style="text-align: left;">생성 수 · 공통 상수</td>
</tr>
<tr>
<td style="text-align: left;"><code>Car.total_created += 1</code></td>
<td style="text-align: left;">공유 값 변경</td>
<td style="text-align: left;"><code>__init__</code> 안에서 개수 세기</td>
</tr>
<tr>
<td style="text-align: left;"><code>self.total_created += 1</code></td>
<td style="text-align: left;">객체에 같은 이름이 새로 생김</td>
<td style="text-align: left;">쓰면 안 됨 (2-3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 클래스 변수는 객체를 만드는 셀을 다시 실행할 때마다 누적됨 → 원본 출력이 127처럼 커진 이유

> ✅ **한 줄 정리** · 클래스 변수는 클래스에 하나만 있어 모든 객체가 같은 값을 보고, 바꿀 때는 `클래스.변수`로 바꾼다.

**딥러닝 코드에서는**

- `class Config: IMG_SIZE = 224` — 모든 곳이 같이 쓰는 상수

### 2-3. `B` 같은 이름 · 삭제 · `isinstance` · `car1.x = 100` · `del` · `isinstance()` <sub>원본 8 ~ 10절</sub>

> 🎯 **객체에 클래스 변수와 같은 이름으로 대입하면 무엇이 바뀌는가?**

💡 **쉽게 말하면**

- 내 사물함에 게시판과 **같은 제목의 쪽지**를 넣은 것.
- 나만 내 쪽지를 보고, 게시판은 그대로.

**실행 흐름**

1. car1.total_created = 100
2. 세 곳 비교 · `__dict__`
3. del 로 삭제
4. isinstance

**먼저 예측** · `car1.total_created = 100` 뒤 `Car.total_created`는? — 답은 코드 실행 결과 아래에

In [6]:
car1.total_created = 100  # 클래스 변수 변경 X → car1에 같은 이름의 인스턴스 변수 생성
# car1만 100
print("car1:", car1.total_created, "/ car2:", car2.total_created, "/ Car:", Car.total_created)
print("car1에 직접 저장된 값:", car1.__dict__)       # total_created가 새로 생김
del car1.total_created                               # car1의 인스턴스 변수 삭제
print("삭제 후 car1:", car1.total_created)           # 못 찾으면 클래스에서 찾음
print(isinstance(car1, Car), isinstance(car2, Car))  # Car로 만든 객체인지

car1: 100 / car2: 5 / Car: 5
car1에 직접 저장된 값: {'company': 'Ferrari', 'color': 'White', 'price': 7500, 'total_created': 100}
삭제 후 car1: 5
True True


**결과** · 예측 정답은 `5` 그대로 — car1에 같은 이름이 새로 생겼을 뿐

- `car1: 100 / car2: 5 / Car: 5` — car1만 100, 클래스 값은 그대로
- `'price': 7500, 'total_created': 100}` — car1의 `__dict__`에 새로 생김
- `삭제 후 car1: 5` · `True True` — 지우면 다시 클래스 값 · 둘 다 Car의 객체

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>객체.변수 = 값</code> (대입)</td>
<td style="text-align: left;">같은 이름의 <strong>인스턴스 변수 생성</strong></td>
<td style="text-align: left;">그 객체만 다른 값을 줄 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>객체.변수</code> (읽기)</td>
<td style="text-align: left;">객체 → 클래스 순서로 찾음</td>
<td style="text-align: left;">읽기는 어디서든</td>
</tr>
<tr>
<td style="text-align: left;"><code>del 객체.변수</code></td>
<td style="text-align: left;">객체의 것만 삭제 → 다시 클래스 값</td>
<td style="text-align: left;">가린 값을 되돌릴 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance(객체, 클래스)</code></td>
<td style="text-align: left;">그 클래스로 만든 객체인지</td>
<td style="text-align: left;">자료형 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 읽기는 클래스까지 찾아가지만 대입은 항상 객체에 새로 만듦 → 공유 값은 꼭 `클래스.변수`로 변경

> ✅ **한 줄 정리** · 객체에 같은 이름으로 대입하면 그 객체에만 인스턴스 변수가 생겨 클래스 변수를 가린다.

**딥러닝 코드에서는**

- `isinstance(model, nn.Module)` — 모델 객체인지 확인

---
## 3. 종합 실습 · nn.Module에서 확인

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - PyTorch 모델 두 개로 self · 인스턴스 변수 · 클래스 변수 확인

<img src="images/pfc2_module.png" alt="모델 두 개가 각자 self.fc를 가지고 클래스 변수는 공유" width="763">

**파이프라인 위치** · ④ 모델

> 🎯 **입력 크기가 다른 `TinyNet` 두 개를 만들어 층 · 출력 · 공유 값을 비교하려면?**

💡 **쉽게 말하면**

- `self.fc`는 **모델마다 자기 층**(인스턴스 변수).
- 모델 개수처럼 같이 쓰는 값은 클래스 변수.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">TinyNet 설계</td>
<td style="text-align: left;">1-1 <code>self</code> · 2-2 클래스 변수</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">모델 두 개 생성</td>
<td style="text-align: left;">2-1 객체마다 층</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">층 · 모양 비교</td>
<td style="text-align: left;">2-1 <code>is</code> · <code>__dict__</code>처럼 확인</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;"><code>model(x)</code> 호출</td>
<td style="text-align: left;">1-1 self = 부른 모델</td>
</tr>
</tbody>
</table>

**먼저 예측** · 모델 두 개의 `self.fc`는 같은 객체일까? — 답은 코드 실행 결과 아래에

In [7]:
import torch          # 텐서
from torch import nn  # 층 · 모델


# 1단계: 설계도 (클래스 변수 1개 + 모델마다 층)
class TinyNet(nn.Module):  # nn.Module을 물려받은 모델
    count = 0              # 클래스 변수: 만든 모델 수

    def __init__(self, n_in):         # 모델마다 입력 크기
        super().__init__()            # nn.Module 준비 (파일별 3에서 자세히)
        self.fc = nn.Linear(n_in, 1)  # 인스턴스 변수 = 이 모델만의 층
        TinyNet.count += 1            # 공유 값은 클래스 이름으로

    def forward(self, x):  # self.fc = 이 모델의 층
        return self.fc(x)  # 입력 → 점수 1개


# 2 ~ 4단계: 모델 두 개를 만들어 비교
torch.manual_seed(0)                                  # 가중치 초기값 고정
m1, m2 = TinyNet(3), TinyNet(5)                       # 모델 두 개
print("층이 같은 객체?", m1.fc is m2.fc)              # 객체마다 따로
# 입력 크기가 달라 모양도 다름
print("가중치 모양:", tuple(m1.fc.weight.shape), tuple(m2.fc.weight.shape))
print("m1 출력:", tuple(m1(torch.ones(2, 3)).shape))  # 배치 2 → 점수 2개
print("만든 모델 수:", TinyNet.count, m1.count)       # 클래스 변수는 공유

층이 같은 객체? False
가중치 모양: (1, 3) (1, 5)
m1 출력: (2, 1)
만든 모델 수: 2 2


**결과** · 예측 정답은 아니요 — `self.fc`는 모델 객체마다 따로인 인스턴스 변수

- `층이 같은 객체? False` — 모델마다 자기 층을 가짐
- `가중치 모양: (1, 3) (1, 5)` · `m1 출력: (2, 1)` — `__init__`에 넘긴 값대로 객체마다 다른 층
- `만든 모델 수: 2 2` — 클래스 변수는 어디서 읽어도 같은 값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>self.fc</code> (인스턴스 변수)</td>
<td style="text-align: left;">모델마다 다른 층 · 가중치</td>
<td style="text-align: left;">층 · 하이퍼파라미터</td>
</tr>
<tr>
<td style="text-align: left;"><code>TinyNet.count</code> (클래스 변수)</td>
<td style="text-align: left;">모든 모델이 공유</td>
<td style="text-align: left;">모델 수 · 공통 상수</td>
</tr>
<tr>
<td style="text-align: left;"><code>m1(x)</code></td>
<td style="text-align: left;"><code>forward(self=m1, x)</code> 실행</td>
<td style="text-align: left;">모델 호출</td>
</tr>
<tr>
<td style="text-align: left;"><code>m1.fc is m2.fc</code></td>
<td style="text-align: left;">False</td>
<td style="text-align: left;">모델마다 층이 따로라는 증거</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 층을 클래스 변수(`fc = nn.Linear(...)`를 `__init__` 밖)로 두면 모든 모델이 같은 층을 공유 → 층은 반드시 `self.`로

> ✅ **한 줄 정리** · `nn.Module`의 층은 `self.층`으로 저장한 인스턴스 변수라 모델마다 따로이고, `model(x)`의 self는 그 모델이다.

**딥러닝 코드에서는**

- 같은 클래스로 만든 모델 두 개를 앙상블해도 가중치가 섞이지 않는 이유

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. self</td>
<td style="text-align: left;"><code>car1.show_info()</code> = <code>Car.show_info(car1)</code></td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2. self 누락</td>
<td style="text-align: left;"><code>takes 0 positional arguments but 1 was given</code></td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3. 인스턴스 변수</td>
<td style="text-align: left;">car1만 7500 · <code>__dict__</code>는 객체마다</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4. 클래스 변수</td>
<td style="text-align: left;">4 → 5 · 어디서 읽어도 같은 값</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 같은 이름 대입</td>
<td style="text-align: left;">car1에만 100 · del 하면 다시 5</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">6. nn.Module</td>
<td style="text-align: left;"><code>m1.fc is m2.fc</code> → False</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>self</code></td>
<td style="text-align: left;">메서드를 부른 객체</td>
<td style="text-align: left;"><code>forward(self, x)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">인스턴스 변수</td>
<td style="text-align: left;"><code>self.x</code> · 객체마다</td>
<td style="text-align: left;"><code>self.fc = nn.Linear(...)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">클래스 변수</td>
<td style="text-align: left;"><code>클래스.x</code> · 공유</td>
<td style="text-align: left;">공통 상수</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">같은 이름 대입</td>
<td style="text-align: left;">객체에 새로 생겨 가림</td>
<td style="text-align: left;">공유 값은 <code>클래스.x</code>로 변경</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>car1.show_info()</code>는 무엇으로 바뀌어 실행되나?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>def hello():</code>를 객체로 부르면?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;"><code>car1.price = 7500</code> 뒤 <code>car2.price</code>는?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">클래스 변수를 바꾸는 올바른 방법은?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;"><code>car1.total_created = 100</code> 뒤 <code>Car.total_created</code>는?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">모델 두 개의 <code>self.fc</code>는 같은가?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `Car("Kia", "Red", 3000)`을 하나 더 만들면 `Car.total_created`는 얼마나 늘어날까?
2. `car2.price = 4000` 뒤 `car3.price`는?

In [8]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `Car`에 클래스 변수 `wheels = 4`를 두고 `car1.wheels`를 읽으면?
2. `car2.wheels = 3` 뒤 `car3.wheels`는?
3. `TinyNet(4)`를 하나 더 만들면 `m1.count`는?

In [9]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>Car.show_info(car1)</code></td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">TypeError — 0개 받는데 1개 전달</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">5000 그대로</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;"><code>Car.total_created += 1</code></td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">그대로 (car1에만 새로 생김)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">아니요 — 모델마다 따로</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
before = Car.total_created
kia = Car("Kia", "Red", 3000)
print(Car.total_created - before)  # 1
car2.price = 4000
print(car3.price)  # 6000
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>4</code> — 객체에 없으면 클래스에서 찾음</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>4</code> — car2에만 인스턴스 변수가 생김</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>3</code> — 클래스 변수는 공유</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>self.total_created += 1</code>로 생성 수를 셈</td>
<td style="text-align: left;">self로 바꾸면 클래스 값이 바뀐다고 착각</td>
<td style="text-align: left;">대입은 객체에 같은 이름을 새로 만듦</td>
<td style="text-align: left;">공유 값을 <code>클래스.변수</code>로 바꿨나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. `self`는 무엇인가요?**
> 메서드를 부른 객체 자신입니다. `car1.show_info()`는 파이썬이 `Car.show_info(car1)`로 바꿔 실행하므로 첫 인자 self에 car1이 들어갑니다. 그래서 메서드는 첫 인자로 self를 받아야 합니다.

**Q2. 인스턴스 변수와 클래스 변수의 차이는 무엇인가요?**
> 인스턴스 변수는 `self.x`로 객체마다 따로 저장되고 클래스 변수는 클래스에 하나만 있어 모든 객체가 공유합니다. 객체에 같은 이름으로 대입하면 클래스 변수가 바뀌지 않고 그 객체에만 새 변수가 생깁니다.

**Q3. PyTorch 모델에서 층을 `self.`로 저장하는 이유는?**
> 층은 모델마다 따로 가중치를 가져야 하므로 인스턴스 변수로 둡니다. `self.fc`로 저장하면 `nn.Module`이 층을 등록해 `parameters()`로 찾을 수 있고, 같은 클래스로 만든 모델 두 개도 층을 공유하지 않습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>self</code> · <code>클래스.메서드(객체)</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">부른 객체 · 직접 넘기기</td>
<td style="text-align: left;">1-1 <code>Car.show_info(car1)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>TypeError</code></td>
<td style="text-align: left;">예외</td>
<td style="text-align: left;">self 누락 · 인자 개수</td>
<td style="text-align: left;">1-2 <code>NoSelf().hello()</code></td>
</tr>
<tr>
<td style="text-align: left;">인스턴스 변수 · <code>__dict__</code></td>
<td style="text-align: left;">Python 기본 문법 · 속성</td>
<td style="text-align: left;">객체마다 값 · 저장된 값</td>
<td style="text-align: left;">2-1 <code>car1.__dict__</code></td>
</tr>
<tr>
<td style="text-align: left;">클래스 변수</td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">모두 공유</td>
<td style="text-align: left;">2-2 <code>Car.total_created += 1</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>del</code> · <code>isinstance()</code></td>
<td style="text-align: left;">Python 기본 문법 · 내장 함수</td>
<td style="text-align: left;">속성 삭제 · 자료형 확인</td>
<td style="text-align: left;">2-3 <code>del car1.total_created</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Module</code> · <code>nn.Linear</code> · <code>torch.manual_seed</code></td>
<td style="text-align: left;">torch.nn · torch</td>
<td style="text-align: left;">모델 · 층 · 초기값 고정</td>
<td style="text-align: left;">3 <code>self.fc = nn.Linear(n_in, 1)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">메서드 첫 인자는 <code>self</code></td>
<td style="text-align: left;"><code>def forward(x):</code></td>
</tr>
<tr>
<td style="text-align: left;">공유 값은 <code>클래스.변수</code>로 변경</td>
<td style="text-align: left;"><code>self.공유값 += 1</code></td>
</tr>
<tr>
<td style="text-align: left;">층은 <code>self.층</code>으로 저장</td>
<td style="text-align: left;">층을 클래스 변수로 두기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">인스턴스 변수 vs 클래스 변수</td>
<td style="text-align: left;">객체마다 vs 공유</td>
<td style="text-align: left;">층은 인스턴스 변수</td>
</tr>
<tr>
<td style="text-align: left;"><code>car1.x</code> 읽기 vs 대입</td>
<td style="text-align: left;">클래스까지 찾음 vs 객체에 새로</td>
<td style="text-align: left;">공유 값은 <code>Car.x =</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>car1.f()</code> vs <code>Car.f(car1)</code></td>
<td style="text-align: left;">자동 vs 직접 self</td>
<td style="text-align: left;">같은 실행</td>
</tr>
<tr>
<td style="text-align: left;"><code>is</code> vs <code>==</code></td>
<td style="text-align: left;">같은 객체 vs 같은 값</td>
<td style="text-align: left;">객체 구분은 <code>is</code></td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">10분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 다시 풀기 · 2-3 출력을 실행 전에 예측</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">3장 TinyNet에 <code>self.n_in</code>을 추가해 모델마다 확인</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 3(8.ipynb) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">10분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 강의노트(`2.ipynb`) 1 ~ 3절을 1장, 4 ~ 10절을 2장으로 나누고 self 누락 · `nn.Module` 실습을 보충
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 그 줄이 왜 필요한가 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1 ~ 3절 객체 생성 · self</td>
<td style="text-align: left;">객체 생성과 self</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">보충 self 누락</td>
<td style="text-align: left;">self를 빠뜨리면</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">4 ~ 5절 인스턴스 변수</td>
<td style="text-align: left;">인스턴스 변수의 독립성</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">6 ~ 7절 클래스 변수</td>
<td style="text-align: left;">클래스 변수</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">8 ~ 10절 같은 이름 · 삭제</td>
<td style="text-align: left;">같은 이름 · 삭제 · isinstance</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">보충 nn.Module</td>
<td style="text-align: left;">nn.Module에서 확인</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [10]:
# ---------- 1-1 ----------
class Car:                                       # 생성 수를 세는 자동차 설계도
    """자동차 한 대의 정보를 표현하는 클래스"""  # 클래스 설명 (docstring)

    total_created = 0  # 클래스 변수: 모든 Car 객체가 함께 쓰는 값

    def __init__(self, company, color, price):  # 객체를 만들 때 자동 실행
        self.company = company                  # 인스턴스 변수: 객체마다 다른 값
        self.color = color                      # 색
        self.price = price                      # 가격
        Car.total_created += 1                  # 공유 값은 클래스 이름으로 변경

    def show_info(self):                                            # self = 이 메서드를 부른 객체
        return f"{self.company} / {self.color} / {self.price:,}원"  # 그 객체의 값으로 문장


car1 = Car("Ferrari", "White", 8000)  # 1번째 생성
car2 = Car("BMW", "Black", 5000)      # 2번째
car3 = Car("Audi", "Silver", 6000)    # 3번째
car4 = Car("Audi", "Silver", 18)      # 원본 1절의 4번째
print(car1.show_info())               # 파이썬이 car1을 self에 넣음
print(Car.show_info(car1))            # 직접 car1을 self 자리에
print("car1 is car2:", car1 is car2)  # 같은 클래스여도 다른 객체
# __str__ 없으면 주소 (실행마다 다름)
print("print(car1) →", str(car1).split(" at ")[0], "at 0x...>")


# ---------- 1-2 ----------
class NoSelf:        # self를 빠뜨린 설계도
    def hello():     # 첫 인자 self 없음
        return "hi"  # 인사


try:
    NoSelf().hello()                       # 객체가 자동으로 첫 인자로 들어감
except TypeError as e:                     # 받을 칸이 없음
    print("TypeError:", e)                 # 에러 문장
print("클래스로 부르면:", NoSelf.hello())  # 객체가 안 넘어가 동작

Ferrari / White / 8,000원
Ferrari / White / 8,000원
car1 is car2: False
print(car1) → <__main__.Car object at 0x...>
TypeError: NoSelf.hello() takes 0 positional arguments but 1 was given
클래스로 부르면: hi


In [11]:
# ---------- 2-1 ----------
print("변경 전:", car1.price, car2.price)  # 8000 5000
car1.price = 7500                          # car1 객체의 값만 바꿈
print("변경 후:", car1.price, car2.price)  # car2는 그대로
print("car1:", car1.__dict__)              # car1에 직접 저장된 값
print("car2:", car2.__dict__)              # car2에 직접 저장된 값


# ---------- 2-2 ----------
print("생성한 자동차 수:", Car.total_created)  # car1 ~ car4
print("car1에서 접근:", car1.total_created)    # 객체로도 읽을 수 있음
car4 = Car("Volvo", "Blue", 4500)              # 7절: 하나 더 생성
print("car4 생성 후:", Car.total_created, "/ car2에서:", car2.total_created)  # 모두 같은 값을 봄


# ---------- 2-3 ----------
car1.total_created = 100  # 클래스 변수 변경 X → car1에 같은 이름의 인스턴스 변수 생성
# car1만 100
print("car1:", car1.total_created, "/ car2:", car2.total_created, "/ Car:", Car.total_created)
print("car1에 직접 저장된 값:", car1.__dict__)       # total_created가 새로 생김
del car1.total_created                               # car1의 인스턴스 변수 삭제
print("삭제 후 car1:", car1.total_created)           # 못 찾으면 클래스에서 찾음
print(isinstance(car1, Car), isinstance(car2, Car))  # Car로 만든 객체인지

변경 전: 8000 5000
변경 후: 7500 5000
car1: {'company': 'Ferrari', 'color': 'White', 'price': 7500}
car2: {'company': 'BMW', 'color': 'Black', 'price': 5000}
생성한 자동차 수: 4
car1에서 접근: 4
car4 생성 후: 5 / car2에서: 5
car1: 100 / car2: 5 / Car: 5
car1에 직접 저장된 값: {'company': 'Ferrari', 'color': 'White', 'price': 7500, 'total_created': 100}
삭제 후 car1: 5
True True


In [12]:
# ---------- 3. 종합 실습 ----------
import torch          # 텐서
from torch import nn  # 층 · 모델


# 1단계: 설계도 (클래스 변수 1개 + 모델마다 층)
class TinyNet(nn.Module):  # nn.Module을 물려받은 모델
    count = 0              # 클래스 변수: 만든 모델 수

    def __init__(self, n_in):         # 모델마다 입력 크기
        super().__init__()            # nn.Module 준비 (파일별 3에서 자세히)
        self.fc = nn.Linear(n_in, 1)  # 인스턴스 변수 = 이 모델만의 층
        TinyNet.count += 1            # 공유 값은 클래스 이름으로

    def forward(self, x):  # self.fc = 이 모델의 층
        return self.fc(x)  # 입력 → 점수 1개


# 2 ~ 4단계: 모델 두 개를 만들어 비교
torch.manual_seed(0)                                  # 가중치 초기값 고정
m1, m2 = TinyNet(3), TinyNet(5)                       # 모델 두 개
print("층이 같은 객체?", m1.fc is m2.fc)              # 객체마다 따로
# 입력 크기가 달라 모양도 다름
print("가중치 모양:", tuple(m1.fc.weight.shape), tuple(m2.fc.weight.shape))
print("m1 출력:", tuple(m1(torch.ones(2, 3)).shape))  # 배치 2 → 점수 2개
print("만든 모델 수:", TinyNet.count, m1.count)       # 클래스 변수는 공유

층이 같은 객체? False
가중치 모양: (1, 3) (1, 5)
m1 출력: (2, 1)
만든 모델 수: 2 2
